# Geomap: Kongruenz nach Zeitphase (Bundesversammlung)

In [ ]:
%load_ext autoreload
%autoreload 2

import importlib
import re
from pathlib import Path

import pandas as pd
import visualisierungen

importlib.reload(visualisierungen)

from visualisierungen import (
    phase_kantons_row_to_map_df,
    schweiz_karte_interaktiv_phasen,
    write_plotly_html_responsive,
)

In [ ]:
PARTEI_BV = "bv-pos_label"

PHASE_TITLES = {
    "phase1_fruehphase": "Frühphase\n(1848-1899)",
    "phase2_volatile": "Volatile Phase\n(1900-1949)",
    "phase3_konsens": "Konsensphase\n(1950-1975)",
    "phase4_aufspaltung": "Aufspaltung\n(1976-2009)",
    "phase5_2010_heute": "2010er–heute\n(2010-heute)",
}


def _phase_order(slug: str) -> int:
    m = re.search(r"phase(\d+)", str(slug))
    return int(m.group(1)) if m else 99


for base in (Path(".."), Path(".")):
    csv_path = base / "data" / "processed" / "df_heatmap_by_phase.csv"
    if csv_path.is_file():
        break
else:
    raise FileNotFoundError("df_heatmap_by_phase.csv nicht gefunden.")

df_phase = pd.read_csv(csv_path)
if "Unnamed: 0" in df_phase.columns:
    df_phase = df_phase.drop(columns=["Unnamed: 0"])

bv = df_phase[df_phase["partei"] == PARTEI_BV]
phase_slugs = sorted(bv["phase"].dropna().unique(), key=_phase_order)
PHASES = [(slug, PHASE_TITLES.get(slug, slug)) for slug in phase_slugs]

phasen_map = []
for slug, titel in PHASES:
    row = bv.loc[bv["phase"] == slug].iloc[0]
    phasen_map.append((titel, phase_kantons_row_to_map_df(row)))

# Akteure für das Dropdown (Reihenfolge wie y-Achse der Heatmap), GLP ausgeschlossen.
AKTEUR_ORDER = [
    "br-pos_label",
    "bv-pos_label",
    "p-gps_label",
    "p-sps_label",
    "p-mitte_label",
    "p-fdp_label",
    "p-svp_label",
]
AKTEUR_LABELS = {
    "br-pos_label": "Bundesrat",
    "bv-pos_label": "Bundesversammlung",
    "p-gps_label": "Grüne",
    "p-sps_label": "SP",
    "p-mitte_label": "Mitte",
    "p-fdp_label": "FDP",
    "p-svp_label": "SVP",
}
DEFAULT_AKTEUR = AKTEUR_LABELS[PARTEI_BV]

akteur_phasen = {}
for partei_slug in AKTEUR_ORDER:
    sub = df_phase[df_phase["partei"] == partei_slug]
    if sub.empty:
        continue
    akteur_phasen[AKTEUR_LABELS.get(partei_slug, partei_slug)] = [
        (titel, phase_kantons_row_to_map_df(sub.loc[sub["phase"] == slug].iloc[0]))
        for slug, titel in PHASES
        if not sub.loc[sub["phase"] == slug].empty
    ]

len(akteur_phasen), list(akteur_phasen)

In [ ]:
fig_bv = schweiz_karte_interaktiv_phasen(
    akteur_phasen=akteur_phasen,
    default_akteur=DEFAULT_AKTEUR,
    height=360,
)
fig_bv.show()
write_plotly_html_responsive(
    fig_bv,
    "../Blog/blog_plots/d7_geomap_zeitphasen_bv.html",
    height=400,
    phase_bar_labels=[titel for titel, _ in phasen_map],
    phase_bar_layout="generic",
)